In [ ]:
import os
import sys
import cv2
import seaborn as sns
import anndata as ad
import argparse
import pandas as pd
import numpy as np
import tacco as tc
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.transforms import Affine2D

In [ ]:
## 截取L2-L6的空间组

In [ ]:
sc_path = "data/processed/excitatory_neuron_file"
sp_dir = "data/processed/spatial_bin100_file"
output_dir = "results/tacco/tacco_output_file"
os.makedirs(output_dir, exist_ok=True)  # 创建输出目录，如果不存在

In [ ]:
# 读取单细胞参考数据
reference = sc.read_h5ad(sc_path)
reference = reference[reference.obs['annotation'] != 'Other']
table = reference.obs["annotation"].value_counts()
prob = reference.obs["annotation"].value_counts() / reference.obs["annotation"].size

In [ ]:
# sp读取
sp_path = "data/processed/spatial_bin100_file"
adata = sc.read_h5ad(sp_path)
adata.X = adata.X.astype('float32')
tc.tl.annotate(adata, reference, "annotation", result_key='pred_celltype', annotation_prior=prob, verbose=False)

In [ ]:
# 提取预测结果
prediction = adata.obsm['pred_celltype'].copy()
prediction['pred_celltype'] = prediction.idxmax(1)
prediction['max_score'] = prediction.iloc[:, :-1].max(1)
pred_prob = prediction['pred_celltype'].value_counts(normalize=True)
mse = ((pred_prob - prob) ** 2).mean()
remove_celltype = pred_prob[pred_prob == 0].index.tolist()
    
# 更新 adata 的注释信息
adata.obs['pred_celltype'] = prediction['pred_celltype']
adata.obs["pred_celltype"] = adata.obs["pred_celltype"].cat.remove_categories(remove_celltype)

In [ ]:
sc.pl.spatial(
        adata,
        color='pred_celltype',
        spot_size=100,
        title=f'Tacco Predicted Cell Types',
        legend_loc='right margin',
        palette='tab20',
        show=True
    )

In [ ]:
## prob modified

In [ ]:
prob

In [ ]:
prob["EX_BDNF"] = 0.06
prob["EX_C1QL3"] = 0.1
prob["EX_ERG"] = 0.1
prob["EX_NTNG1"] = 0.1

prob["EX_HTR7_GNAL"] = 0.15
prob["EX_FOXP2"] = 0.1
prob["EX_ITGA4_TPBG"] = 0.1

prob["EX_env_TMEM144"] = 0.06
prob["EX_SLC1A3_FGFR3"] = 0.05
prob["EX_Ntng2"] = 0.04
prob["EX_ADRA1A_Vat1l"] = 0.04
prob["EX_TSHZ2_VWC2L"] = 0.04
prob["EX_Nxph4_CPLX3"] = 0.04
prob["EX_IFITM3_CLDN5"] = 0.02

In [ ]:
## 修改后重新预测

In [ ]:
sp_path = "data/processed/spatial_bin100_file"
adata = sc.read_h5ad(sp_path)
adata.X = adata.X.astype('float32')
tc.tl.annotate(adata, reference, "annotation", result_key='pred_celltype', annotation_prior=prob, verbose=False)

In [ ]:
# 提取预测结果
prediction = adata.obsm['pred_celltype'].copy()
prediction['pred_celltype'] = prediction.idxmax(1)
prediction['max_score'] = prediction.iloc[:, :-1].max(1)
pred_prob = prediction['pred_celltype'].value_counts(normalize=True)
mse = ((pred_prob - prob) ** 2).mean()
remove_celltype = pred_prob[pred_prob == 0].index.tolist()
    
# 更新 adata 的注释信息
adata.obs['pred_celltype'] = prediction['pred_celltype']
adata.obs["pred_celltype"] = adata.obs["pred_celltype"].cat.remove_categories(remove_celltype)

In [ ]:
sc.pl.spatial(
        adata,
        color='pred_celltype',
        spot_size=100,
        title=f'Tacco Predicted Cell Types',
        legend_loc='right margin',
        palette='tab20',
        show=True
    )

In [ ]:
adata.write("results/tacco/tacco_output_file")

In [ ]:
## 预测概率热图

In [ ]:
input_path = "results/tacco/tacco_output_file"

h5ad_files = [f for f in os.listdir(input_path) if f.endswith('.h5ad')]
for file in h5ad_files:
    chip_name = file.split('_')[0]
    adata = sc.read_h5ad(os.path.join(input_path, file))
#    sc.set_figure_params(figsize=(8, 6), dpi=100)
    clusters = ['EX_NTNG1', 'EX_BDNF']
    for cluster in clusters:
        if cluster in adata.obsm['pred_celltype'].columns:
            adata.obs[f'{cluster}_prob'] = adata.obsm['pred_celltype'][cluster]
            sc.pl.embedding(
                adata,
                basis='spatial',  # 指定嵌入的坐标
                color=f'{cluster}_prob',
                size=15,
                legend_loc='right margin',  # 将图例放在右侧
                legend_fontsize=12,  # 设置图例字体大小
                frameon=False,  # 去掉坐标框
                title='',  # 去掉标题
                show=False,
                save=f'{cluster}_{chip_name}_prob.png'
            )
            sc.pl.embedding(
                adata,
                basis='spatial',  # 指定嵌入的坐标
                color=f'{cluster}_prob',
                size=15,
                legend_loc='right margin',  # 将图例放在右侧
                legend_fontsize=12,  # 设置图例字体大小
                frameon=False,  # 去掉坐标框
                title='',  # 去掉标题
                show=False,
                save=f'{cluster}_{chip_name}_prob.pdf'
            )
        else:
            print(f"Warning: {cluster} not found in {file}")

In [ ]:
## z-score中位值分布热图

In [ ]:
input_path = "results/tacco/tacco_output_file"
output_path = "results/tacco/tacco_output_file"

h5ad_files = [f for f in os.listdir(input_path) if f.endswith('.h5ad')]
clusters = ['EX_NTNG1', 'EX_BDNF']

heatmap_data = {cluster: pd.DataFrame() for cluster in clusters}

chip_name_map = {
    "D04300F4": "SA-1-PFC",
    "Y00860K2": "SA-1-V1",
    "Y00783L8": "SA-2-V1",
    "A03589G3": "SA-2-PFC",
    "Y00789G7": "NA-1-PFC",
    "Y00861G1": "NA-2-V1",
    "Y00782J8": "NA-1-V1",
    "D03058F3": "SB-1-PFC",
    "A02999G6": "SB-1-V1",
    "Y00860K3": "BU-1-PFC"
}

for file in h5ad_files:
    chip_name = file.split('_')[0]
    adata = sc.read_h5ad(os.path.join(input_path, file))

    for cluster in clusters:
        if cluster in adata.obsm['pred_celltype'].columns:
            adata.obs[f'{cluster}_prob'] = adata.obsm['pred_celltype'][cluster]
            df = adata.obs.dropna(subset=[f'{cluster}_prob']).copy()
            summary = df.groupby('region')[f'{cluster}_prob'].median().reset_index()
            # 2. 在这里替换
            summary['chip_name'] = chip_name_map.get(chip_name, chip_name)
            summary['cluster'] = cluster
            heatmap_data[cluster] = pd.concat([heatmap_data[cluster], summary], ignore_index=True)
        else:
            print(f"Warning: {cluster} not found in {file}")

# 设置全局字体大小
plt.rcParams.update({'font.size': 14})

for cluster in clusters:
    heatmap_data_pivot = heatmap_data[cluster].pivot_table(index='region', columns='chip_name', values=f'{cluster}_prob')
    # 计算按列的 z-score
    heatmap_data_zscore = heatmap_data_pivot.apply(zscore, axis=0)
    
    # 绘制热图
    plt.figure(figsize=(15, 6))
    sns.heatmap(heatmap_data_zscore, cmap='viridis', annot=True, fmt=".2f", linewidths=.5, annot_kws={"fontsize": 14})
    
    # 去掉坐标轴的标签和标题
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.xlabel('')
    plt.ylabel('')
    plt.title('')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f'heatmap_{cluster}_zscore_median_prob.png'))
    plt.close()